# Graded Activity: Let's revisit the flow problem as a linear algebra system
In this activity, we'll reformulate the flow problem as a linear algebra system and solve it using our iterative solvers and the QR iteration method.

__Learning objectives:__ Fill me in.

Let's go!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's setup our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl"));

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types and data used in this material. 

### Constants
Let's define some constants that will be used throughout the notebook. See the comment for a description of each constant, what it represents, its value, units, etc.

In [2]:
# We are going to plot the path through a graph, so let's provide the coordinates for each node, i.e., the layout
# This layout looks like our schematic but you can rearrange this if you want!
node_coordinates = [

    # warehouse node (you)
    10.0 10.0 ; # 1 warehouse node s (x,y) coordinates

    # processing nodes (workers)
    11.0 11.0 ; # 2 incoming shipping processing node (x,y) coordinates
    11.0 10.0 ; # 3 incoming shipping processing node (x,y) coordinates
    11.0 9.0 ; # 4 incoming shipping processing node (x,y) coordinates

    # job nodes (tasks)
    12.0 11.0 ; # 5 job node (x,y) coordinates
    12.0 10.0 ; # 6 job node (x,y) coordinates
    12.0 9.0 ; # 7 job node (x,y) coordinates
    12.0 8.0 ; # 8 job node (x,y) coordinates

    # sink nodes (targets, tasks done!)
    13.0 11.0 ; # 9 sink node t (x,y) coordinates
    13.0 10.0 ; # 10 sink node t (x,y) coordinates
    13.0 9.0 ; # 11 sink node t (x,y) coordinates
    13.0 8.0 ; # 12 sink node t (x,y) coordinates
    
    14.0 10.0 ; # 13 end node (x,y) coordinates
];

___

## Task 1: Build a production process graph model
In this task, we'll build a graph model for our production process. We'll use this model to compute a constraint matrix $\mathbf{A}$.

The problem graph edges are stored in `data/Production-Process-Bipartite.edgelist` with fields: 

> __Records__: Each record in our edgelist file has the comma separated fields: `source,` `target,` `cost,` `lb capacity,` `ub capacity`. The `source` field is the id for the source node, e.g., `1`, the `target` field is the target node id, the `cost` is the cost of assigning the source node to the target node, the `lb capacity` is the lower bound capacity for the edge, and the `ub capacity` is the upper bound capacity for the edge.

In this activity, we'll ignore the edge weight (set to `1`), and instead will focus on the capacity constraints. Ok, so now let's setup our edge parser __callback function__:

In [3]:
"""
    function edgerecordparser(record::String, delim::Char=',') -> Tuple{Int, Int, Float64} | Nothing

This method is called to parse a single edge record from the edgelist file. It gets called once for each record in the file. 
The function splits the record into fields based on the specified delimiter and extracts the source node, target node, and cost (weight) of the edge. 
It returns a tuple containing these values. If the record does not have the expected number of fields, it returns `nothing`.

### Arguments
- `record`: The edge record string to parse.
- `delim`: The delimiter used to split the record.

### Returns
- A tuple containing the source node, target node, and cost of the edge, or `nothing` if the record is invalid.
"""
function edgerecordparser(record::String, delim::Char=',')
    
    # record (five fields)
    # source, target, cost, lb, ub

    fields = split(record, delim) # this assumes a record of the form "source,target,weight"
    if length(fields) < 5 # we have 5 fields
        return nothing
    end

    # get my data from the line -
    source = parse(Int, fields[1]) # source id
    target = parse(Int, fields[2]) # target id
    cost = parse(Float64, fields[3]) # edge weight
    l = parse(Float64, fields[4]) # lower bound capacity
    u = parse(Float64, fields[5]) # upper bound capacity

    # return a tuple -
    return (source, target, cost, l, u)
end;

Next, let's set the path to the edge list file in the `path_to_edge_file::String` variable:

In [4]:
path_to_edge_file = joinpath(_PATH_TO_DATA, "Production-Process-Bipartite.edgelist"); # this points to the graph shown above

Next, construct a dictionary [of `MyConstrainedGraphEdgeModel` instances](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MyConstrainedGraphEdgeModel) which stores the data for the edges. Let's save our edge models in the `myedgemodels::Dict{Int64, MyConstrainedGraphEdgeModel}` dictionary.

The keys in the edge dictionary will be the edge ids (which we can assume are unique), and the values will be the corresponding `MyConstrainedGraphEdgeModel` instances. Here, we've used the line index in the edgefile as the edge id.

In [5]:
myedgemodels = MyConstrainedGraphEdgeModels(path_to_edge_file, edgerecordparser, delim=',', comment='#')

Dict{Int64, MyConstrainedGraphEdgeModel} with 23 entries:
  5  => MyConstrainedGraphEdgeModel(5, 7, 11, 1.0, 0.0, 1.0)
  16 => MyConstrainedGraphEdgeModel(16, 3, 6, 1.0, 0.0, 1.0)
  20 => MyConstrainedGraphEdgeModel(20, 4, 6, 1.0, 0.0, 1.0)
  12 => MyConstrainedGraphEdgeModel(12, 2, 6, 1.0, 0.0, 1.0)
  8  => MyConstrainedGraphEdgeModel(8, 10, 13, 1.0, 0.0, 1.0)
  17 => MyConstrainedGraphEdgeModel(17, 3, 7, 1.0, 0.0, 1.0)
  1  => MyConstrainedGraphEdgeModel(1, 1, 3, 1.0, 0.0, 1.0)
  19 => MyConstrainedGraphEdgeModel(19, 4, 5, 1.0, 0.0, 1.0)
  0  => MyConstrainedGraphEdgeModel(0, 1, 2, 1.0, 0.0, 1.0)
  22 => MyConstrainedGraphEdgeModel(22, 4, 8, 1.0, 0.0, 1.0)
  6  => MyConstrainedGraphEdgeModel(6, 8, 12, 1.0, 0.0, 1.0)
  11 => MyConstrainedGraphEdgeModel(11, 2, 5, 1.0, 0.0, 1.0)
  9  => MyConstrainedGraphEdgeModel(9, 11, 13, 1.0, 0.0, 1.0)
  14 => MyConstrainedGraphEdgeModel(14, 2, 8, 1.0, 0.0, 1.0)
  3  => MyConstrainedGraphEdgeModel(3, 5, 9, 1.0, 0.0, 1.0)
  7  => MyConstrainedGraphEd

Finally, we can build a graph instance. Since this is a directed graph, we'll construct [a `MyDirectedBipartiteGraphModel` instance](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MyDirectedBipartiteGraphModel) using [a `build(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/#VLDataScienceMachineLearningPackage.build). Let's save our graph model in the `directedgraphmodel::MyDirectedBipartiteGraphModel` variable.

In [6]:
directedgraphmodel = let

    # initialize -
    s = 1; # what is the source node
    t = 13; # what is the sink node

    # call the build method to create the graph model
    model = build(MyDirectedBipartiteGraphModel, (
        s = s, # source index
        t = t, # sink index
        edges = myedgemodels
    ));

    
    model # return the model 
end;

In [7]:
directedgraphmodel |> typeof |> T-> fieldnames(T)

(:nodes, :edges, :children, :edgesinverse, :left, :right, :source, :sink, :capacity)

Let's build a map between the $(u,v)$ pairs and their corresponding edge indices. We'll call this the `edge_index_map::Dict{Tuple{Int,Int},Int}`.


In [8]:
edge_index_map = let

    # initialize -
    edge_index_map = Dict{Tuple{Int,Int},Int}() # (u,v) -> edge index
    for (k, (u,v)) ∈ directedgraphmodel.edgesinverse
        edge_index_map[(u,v)] = k
    end
    edge_index_map # return
end

Dict{Tuple{Int64, Int64}, Int64} with 23 entries:
  (4, 5)   => 12
  (1, 2)   => 1
  (6, 10)  => 17
  (8, 12)  => 19
  (12, 13) => 23
  (2, 5)   => 4
  (1, 3)   => 2
  (3, 7)   => 10
  (5, 9)   => 16
  (3, 8)   => 11
  (1, 4)   => 3
  (2, 6)   => 5
  (4, 6)   => 13
  (11, 13) => 22
  (9, 13)  => 20
  (4, 7)   => 14
  (2, 7)   => 6
  (4, 8)   => 15
  (2, 8)   => 7
  ⋮        => ⋮

### System Matrix 
Let's build a system matrix $\mathbf{A}$ for our graph model.

In [9]:
A = let

    # initialize -
    edges = directedgraphmodel.edges; # get the edges
    nodes = directedgraphmodel.nodes; # get the nodes
    edgesinverse = directedgraphmodel.edgesinverse; # get the inverse edges
    d = length(edges); # how many edges are there?
    n = length(nodes); # how many nodes are there?
    A = zeros(n, d); # initialize the system matrix

    # fill the system matrix
    for (k,v) ∈ edgesinverse
        
        w = edges[v]; # weight of (u,v) edges
        A[v[1], k] = -w;
        A[v[2], k] = w;
    end

    A; # return
end

13×23 Matrix{Float64}:
 -1.0  -1.0  -1.0   0.0   0.0   0.0  …   0.0   0.0   0.0   0.0   0.0   0.0
  1.0   0.0   0.0  -1.0  -1.0  -1.0      0.0   0.0   0.0   0.0   0.0   0.0
  0.0   1.0   0.0   0.0   0.0   0.0      0.0   0.0   0.0   0.0   0.0   0.0
  0.0   0.0   1.0   0.0   0.0   0.0      0.0   0.0   0.0   0.0   0.0   0.0
  0.0   0.0   0.0   1.0   0.0   0.0      0.0   0.0   0.0   0.0   0.0   0.0
  0.0   0.0   0.0   0.0   1.0   0.0  …   0.0   0.0   0.0   0.0   0.0   0.0
  0.0   0.0   0.0   0.0   0.0   1.0     -1.0   0.0   0.0   0.0   0.0   0.0
  0.0   0.0   0.0   0.0   0.0   0.0      0.0  -1.0   0.0   0.0   0.0   0.0
  0.0   0.0   0.0   0.0   0.0   0.0      0.0   0.0  -1.0   0.0   0.0   0.0
  0.0   0.0   0.0   0.0   0.0   0.0      0.0   0.0   0.0  -1.0   0.0   0.0
  0.0   0.0   0.0   0.0   0.0   0.0  …   1.0   0.0   0.0   0.0  -1.0   0.0
  0.0   0.0   0.0   0.0   0.0   0.0      0.0   1.0   0.0   0.0   0.0  -1.0
  0.0   0.0   0.0   0.0   0.0   0.0      0.0   0.0   1.0   1.0   1.0   1.0

Let's check a few nodes (rows) of the system matrix $\mathbf{A}$ to make sure it's constructed correctly. Consider two cases:

> __Test cases__
> 
> __Case 1:__ the source node `1` should have zero flows in, and an outgoing edge at nodes `2`, `3`, `4` where the coefficient $a_{ij}$ will be the negative edge weight $-w_{ij}$ (it is negative becaause it is leaving node 1).
>
> __Case 2:__  Any node not equal to `1` or `13` should have both incoming and outgoing edges, where the coefficients $a_{ij}$ will be the edge weights $w_{ij}$ for incoming edges and the negative edge weights $-w_{ij}$ for outgoing edges.

Let's check case 1:

In [10]:
let
    
    # initialize -
    test_node_index = 1; # we are looking at the source node 1
    test_row = A[test_node_index, :]; # row for node 1 (the source node);
    test_children_set = directedgraphmodel.children[test_node_index]; # children of node 1
    test_children_array = test_children_set |> collect |> sort; # convert to sorted array

    # test the edge to each of test_node_index children
    for i ∈ test_children_array
        edge_coordinates = (test_node_index, i);
        edge_weight = directedgraphmodel.edges[edge_coordinates];
        edge_index = edge_index_map[edge_coordinates];
        @assert test_row[edge_index] == -edge_weight "Error: edge weight mismatch at node $test_node_index to child $i"
    end

    # all tests passed
    println("All tests passed!")
end

All tests passed!


Next, let's check case 2. 

In [11]:
let
    
    # initialize -
    test_node_index = 2; # we are looking at a source node (any node not 1 or 13)
    test_row = A[test_node_index, :]; # row for node 1 (the source node);
    test_children_set = directedgraphmodel.children[test_node_index]; # children of node 1
    test_children_array = test_children_set |> collect |> sort; # convert to sorted array

    # find the nonzero elements of the test_row -
    test_nonzero_edge_indices = findall(!iszero, test_row);
    for i ∈ test_nonzero_edge_indices
        
        # get data for this edge -
        edge_coordinates = directedgraphmodel.edgesinverse[i]; # the (u,v) value for this edge index
        edge_weight = directedgraphmodel.edges[edge_coordinates]; # the weight of this edge

        

        # testing logic -
        if edge_coordinates[1] == test_node_index # this is an outgoing edge
            @assert test_row[i] == -edge_weight "Error: edge weight mismatch at node $test_node_index to child $(edge_coordinates[2])"
        elseif edge_coordinates[2] == test_node_index # this is an incoming edge
            @assert test_row[i] == edge_weight "Error: edge weight mismatch at node $test_node_index from parent $(edge_coordinates[1])"
        else
            error("Error: edge index $i does not correspond to node $test_node_index")
        end
    end
end

### Setup the right-hand side vector
Now that we have the system matrix $\mathbf{A}$ set up, we need to create the right-hand side vector $\mathbf{b}$. This vector will represent the net flow at each node, which is the difference between the inflow and outflow for that node.

There is a lot that we can do by setting the elements of this vector, however, for now let's say we want to set the flow into the source node (node `1`) to be equal to the total demand at the sink node (node `13`).


In [19]:
b = let

    # initialize -
    nodes = directedgraphmodel.nodes; # get the nodes
    number_of_nodes = length(nodes);
    b = zeros(number_of_nodes); # initialize the right-hand side vector

    # set the flow into the source node (node 1) to be equal to the total demand at the sink node (node 13)
    b[1] = -3.0; # pick a value for the flow into the source node

    b # return 
end

13-element Vector{Float64}:
 -3.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0

## Task 2: Solve the system using QR deomposition
In this task, we'll solve the linear system $\mathbf{A}\mathbf{v} = \mathbf{b}$ using the QR decomposition method.

Fill me in.

In [ ]:
v̂ = let

    


    # do the decomposition -
    (Q,R) = qr(A);
    v = transpose(R)*transpose(Q)*b; # solve R*v = Q'*b

    v # return
end

23-element Vector{Float64}:
  2.999999999999999
  2.9999999999999982
  2.9999999999999982
  6.661338147750939e-16
  3.3306690738754696e-16
  1.3322676295501878e-15
  6.661338147750939e-16
 -4.3034305114333264e-16
  9.810899124448752e-18
  1.350353932140771e-16
  ⋮
 -7.553920215793915e-16
  4.4674793957957916e-17
 -2.1284927075857607e-16
  5.2398678133831695e-17
  1.2234717449844482e-16
  0.0
  0.0
  0.0
  0.0

Fill me in

In [21]:
let
    residual = A*v̂ - b
end

13-element Vector{Float64}:
 -5.9999999999999964
  2.9999999999999956
  2.9999999999999982
  3.0000000000000004
 -2.330974388676697e-16
  1.642210622787707e-16
  8.913547485634268e-16
  4.76510876322546e-17
  4.4674793957957916e-17
 -2.1284927075857607e-16
  5.2398678133831695e-17
  1.2234717449844482e-16
  0.0

## Task 3: Solve the flow problem system using iteration
In this task, we'll solve the flow problem system using the iterative methods we developed in this module.

## Summary
Fill me in.